# Stage 7 — Dataset Diagnostic Audit

This notebook investigates plausible dataset-side mechanisms behind the frozen Stage 6 cross-domain failures.

**No model loading, inference, training, adaptation, or threshold tuning is performed.**

In [ ]:
# Cell 1 — Mount Drive and locate the frozen data bundle
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE=Path('/content/drive/MyDrive')
OUT=DRIVE/'Stage7_Dataset_Diagnostic_Audit'
OUT.mkdir(parents=True, exist_ok=True)
PARTS=[
    DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_00',
    DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_01',
    DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_02',
]
missing=[str(p) for p in PARTS if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing frozen-data bundle part(s): '+', '.join(missing))
print('STAGE7_AUDIT_ENVIRONMENT: PASS')

In [ ]:
# Cell 2 — Reassemble and verify the frozen bundle
import hashlib, zipfile, shutil
BUNDLE=Path('/content/Stage5_Frozen_h9_v2_Data_Bundle.zip')
EXPECTED_SHA='dd8ce20f2650848133ce4323b00139684b0d4ac8b7db618e8df469244221cee9'
h=hashlib.sha256()
with BUNDLE.open('wb') as out:
    for part in PARTS:
        with part.open('rb') as f:
            for chunk in iter(lambda:f.read(8*1024*1024),b''):
                out.write(chunk); h.update(chunk)
assert h.hexdigest()==EXPECTED_SHA, (h.hexdigest(),EXPECTED_SHA)
DATA_ROOT=Path('/content/stage7_frozen_data')
if DATA_ROOT.exists(): shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True)
with zipfile.ZipFile(BUNDLE) as z:
    assert z.testzip() is None
    z.extractall(DATA_ROOT)
Y_DATA=DATA_ROOT/'h9_v2/Y-PPE-h9-v2/data.yaml'
C_DATA=DATA_ROOT/'h9_v2/Construction-PPE-h9-v2/data.yaml'
assert Y_DATA.is_file() and C_DATA.is_file()
print('STAGE7_FROZEN_DATA: PASS')

In [ ]:
# Cell 3 — Clone Stage 7 analysis branch and install audit dependencies
import subprocess, shutil, sys
REPO=Path('/content/Y-PPE-Detection-RFDETR')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run([
    'git','clone','--branch','stage7-results-analysis','--single-branch',
    'https://github.com/EZZALDEEN049/Y-PPE-Detection-RFDETR.git',str(REPO)
],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pandas','Pillow','PyYAML==6.0.3'],check=True)
print('STAGE7_CODE_AND_DEPS: PASS')

In [ ]:
# Cell 4 — Execute dataset diagnostic audit
import subprocess, sys
AUDIT=REPO/'cross_domain_reliability/stage7_results_analysis/diagnostic_dataset_audit.py'
subprocess.run([
    sys.executable,str(AUDIT),
    '--y-data',str(Y_DATA),
    '--c-data',str(C_DATA),
    '--out',str(OUT),
],check=True)

In [ ]:
# Cell 5 — Display the key diagnostic tables
import pandas as pd
from IPython.display import display
print('Class instance counts')
display(pd.read_csv(OUT/'stage7_class_instance_counts.csv'))
print('\nSafety-class support')
display(pd.read_csv(OUT/'stage7_safety_class_support.csv'))
print('\nBounding-box area summary')
display(pd.read_csv(OUT/'stage7_bbox_area_summary.csv'))
print('\nImage summary')
display(pd.read_csv(OUT/'stage7_image_summary.csv'))
print('\nClass imbalance summary')
display(pd.read_csv(OUT/'stage7_class_imbalance_summary.csv'))